# FT-Transformer

In [1]:
!python -m pip install --upgrade pip setuptools wheel

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
!pip install ipywidgets

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [3]:
!pip install rtdl_revisiting_models -q

In [4]:
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from rtdl_revisiting_models import FTTransformer

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
DATA_PATH = "FeatureB_Repeated"
OUTPUT_PATH = "Official_FTTransformer_FeatureB_Results"


os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

cuda


In [6]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [7]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            * (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc


def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [9]:
def stratified_sample_binary(df, sample_size, random_seed):
    if sample_size >= len(df):
        return df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    sample_ratio = sample_size / len(df)

    sampled_df = (
        df
        .groupby("label", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=max(1, int(len(x) * sample_ratio)),
                random_state=random_seed
            )
        )
        .sample(frac=1, random_state=random_seed)
        .reset_index(drop=True)
    )

    return sampled_df

In [10]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)

    train_indices = []
    test_indices = []

    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [11]:
def train_official_ft_transformer(
    train_df,
    test_df,
    lr=1e-4,
    weight_decay=1e-5,
    batch_size=4096,
    epochs=3
):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values

    X_test = test_df[feature_cols].values
    y_test = test_df["label"].values

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = FTTransformer(
        n_cont_features=len(feature_cols),
        cat_cardinalities=[],
        d_out=1,
        **FTTransformer.get_default_kwargs()
    ).to(device)

    optimizer = model.make_default_optimizer()
    
    # Override default optimizer lr / weight_decay if needed
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = weight_decay

    criterion = nn.BCEWithLogitsLoss()

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, None).squeeze(1)

            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1} loss:", round(total_loss, 4))

    model.eval()

    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X, None).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).int()

            all_preds.extend(preds.detach().cpu().tolist())
            all_scores.extend(probs.detach().cpu().tolist())
            all_labels.extend(batch_y.detach().cpu().tolist())

    metrics = compute_metrics_from_scratch(
        all_labels,
        all_preds,
        all_scores
    )

    return metrics

In [12]:
TRAIN_SAMPLE_SIZE = 30000
TEST_SAMPLE_SIZE = 10000

LR_VALUES = [5e-5, 1e-4, 5e-4]
WEIGHT_DECAY_VALUES = [1e-5]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2

BATCH_SIZE = 1024
EPOCHS = 3

all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):
    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_B_train.csv")
    test_path = os.path.join(repeat_folder, "feature_B_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = stratified_sample_binary(
        train_df,
        sample_size=TRAIN_SAMPLE_SIZE,
        random_seed=42 + repeat_id
    )

    test_df = stratified_sample_binary(
        test_df,
        sample_size=TEST_SAMPLE_SIZE,
        random_seed=100 + repeat_id
    )

    print("Outer train shape:", train_df.shape)
    print("Outer test shape:", test_df.shape)

    best_lr = None
    best_weight_decay = None
    best_mean_val_f1 = -1

    for lr in LR_VALUES:
        for weight_decay in WEIGHT_DECAY_VALUES:

            inner_f1_scores = []

            for inner_id in range(N_INNER_REPEATS):
                inner_train_df, val_df = stratified_split_from_scratch(
                    train_df,
                    label_col="label",
                    test_ratio=VALID_RATIO,
                    random_seed=2000 + repeat_id * 10 + inner_id
                )

                print(
                    f"Trying lr={lr}, weight_decay={weight_decay}, inner={inner_id + 1}"
                )

                val_metrics = train_official_ft_transformer(
                    train_df=inner_train_df,
                    test_df=val_df,
                    lr=lr,
                    weight_decay=weight_decay,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS
                )

                inner_f1_scores.append(val_metrics["f1"])

            mean_val_f1 = np.mean(inner_f1_scores)
            std_val_f1 = np.std(inner_f1_scores, ddof=1)

            all_tuning_results.append({
                "outer_repeat": repeat_id,
                "lr": lr,
                "weight_decay": weight_decay,
                "mean_validation_f1": mean_val_f1,
                "std_validation_f1": std_val_f1
            })

            print("Mean validation F1:", round(mean_val_f1, 6))

            if mean_val_f1 > best_mean_val_f1:
                best_mean_val_f1 = mean_val_f1
                best_lr = lr
                best_weight_decay = weight_decay

    print("Best lr:", best_lr)
    print("Best weight_decay:", best_weight_decay)
    print("Best mean validation F1:", best_mean_val_f1)

    test_metrics = train_official_ft_transformer(
        train_df=train_df,
        test_df=test_df,
        lr=best_lr,
        weight_decay=best_weight_decay,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS
    )

    final_result = {
        "repeat": repeat_id,
        "best_lr": best_lr,
        "best_weight_decay": best_weight_decay,
        "batch_size": BATCH_SIZE,
        "best_mean_val_f1": best_mean_val_f1,
        **test_metrics
    }

    all_results.append(final_result)

    print("Accuracy :", round(test_metrics["accuracy"], 4))
    print("Precision:", round(test_metrics["precision"], 4))
    print("Recall   :", round(test_metrics["recall"], 4))
    print("F1       :", round(test_metrics["f1"], 4))
    print("AUC      :", round(test_metrics["auc"], 4))

Outer Repeat 01


/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 122)
Outer test shape: (9999, 122)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 12.0815
Epoch 2 loss: 8.8415
Epoch 3 loss: 8.1996
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.0488
Epoch 2 loss: 9.0686
Epoch 3 loss: 8.2515
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.3983
Epoch 2 loss: 8.859
Epoch 3 loss: 8.1044
Mean validation F1: 0.84577
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.3631
Epoch 2 loss: 8.3708
Epoch 3 loss: 7.7817
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.1164
Epoch 2 loss: 8.5089
Epoch 3 loss: 7.6984
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.2595
Epoch 2 loss: 8.8302
Epoch 3 loss: 7.9607
Mean validation F1: 0.852409
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.6924
Epoch 2 loss: 8.0003
Epoch 3 loss: 7.4354
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.3153
Epoch 2 loss: 7.885
Epoch 3 loss: 7.2889
Trying

/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 122)
Outer test shape: (9999, 122)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 12.294
Epoch 2 loss: 8.8289
Epoch 3 loss: 8.0818
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.4008
Epoch 2 loss: 8.8521
Epoch 3 loss: 8.2454
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.4911
Epoch 2 loss: 8.8004
Epoch 3 loss: 8.1186
Mean validation F1: 0.854743
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.767
Epoch 2 loss: 8.579
Epoch 3 loss: 7.8078
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.7603
Epoch 2 loss: 9.0932
Epoch 3 loss: 8.1052
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 11.1497
Epoch 2 loss: 8.254
Epoch 3 loss: 7.5709
Mean validation F1: 0.852994
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.5775
Epoch 2 loss: 7.7557
Epoch 3 loss: 7.3799
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.7656
Epoch 2 loss: 8.2853
Epoch 3 loss: 7.5341
Trying 

...

Outer train shape: (29999, 122)
Outer test shape: (9999, 122)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.0068
Epoch 2 loss: 9.0341
Epoch 3 loss: 8.2059
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.6019
Epoch 2 loss: 8.7736
Epoch 3 loss: 8.1549
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 13.0916
Epoch 2 loss: 9.1328
Epoch 3 loss: 8.3918
Mean validation F1: 0.842754
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.9561
Epoch 2 loss: 8.5344
Epoch 3 loss: 7.7796
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.7609
Epoch 2 loss: 8.15
Epoch 3 loss: 7.6618
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 10.8957
Epoch 2 loss: 8.1975
Epoch 3 loss: 7.6911
Mean validation F1: 0.850356
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.8158
Epoch 2 loss: 8.2293
Epoch 3 loss: 7.432
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.8686
Epoch 2 loss: 8.2225
Epoch 3 loss: 7.4838
Trying

/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 122)
Outer test shape: (9999, 122)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.5946
Epoch 2 loss: 9.5552
Epoch 3 loss: 8.6575
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.839
Epoch 2 loss: 9.3263
Epoch 3 loss: 8.3986
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.2737
Epoch 2 loss: 8.8397
Epoch 3 loss: 8.1012
Mean validation F1: 0.842945
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 12.2377
Epoch 2 loss: 8.6482
Epoch 3 loss: 7.866
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.1271
Epoch 2 loss: 8.2024
Epoch 3 loss: 7.6081
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 11.6662
Epoch 2 loss: 8.4211
Epoch 3 loss: 7.7024
Mean validation F1: 0.85096
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 12.1644
Epoch 2 loss: 8.2483
Epoch 3 loss: 7.5656
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.9607
Epoch 2 loss: 8.0375
Epoch 3 loss: 7.468
Trying 

/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_8498/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 122)
Outer test shape: (9999, 122)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.8495
Epoch 2 loss: 9.1408
Epoch 3 loss: 8.2294
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.3097
Epoch 2 loss: 8.9522
Epoch 3 loss: 8.2013
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.5451
Epoch 2 loss: 8.7794
Epoch 3 loss: 8.0659
Mean validation F1: 0.851296
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.5822
Epoch 2 loss: 8.4552
Epoch 3 loss: 7.7959
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.9733
Epoch 2 loss: 8.3569
Epoch 3 loss: 7.6861
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 11.6138
Epoch 2 loss: 8.2844
Epoch 3 loss: 7.656
Mean validation F1: 0.852116
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.1408
Epoch 2 loss: 8.2403
Epoch 3 loss: 7.5345
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.5207
Epoch 2 loss: 7.8412
Epoch 3 loss: 7.3689
Tryi

In [13]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.DataFrame(all_tuning_results)

results_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureB_repeated_results.csv"
)

tuning_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureB_tuning_results.csv"
)

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
Official_FTTransformer_FeatureB_Results/FTTransformer_FeatureB_repeated_results.csv
Saved tuning results to:
Official_FTTransformer_FeatureB_Results/FTTransformer_FeatureB_tuning_results.csv


,repeat,best_lr,best_weight_decay,batch_size,best_mean_val_f1,accuracy,precision,recall,f1,auc,tp,tn,fp,fn
0,1,0.00010,0.00001,1024,0.852409,0.644364,0.633798,0.683010,0.657484,0.660102,3413,3030,1972,1584
1,2,0.00050,0.00001,1024,0.855173,0.618662,0.635780,0.554733,0.592498,0.636152,2772,3414,1588,2225
2,3,0.00010,0.00001,1024,0.848638,0.646665,0.634956,0.689214,0.660973,0.667483,3444,3022,1980,1553
3,4,0.00005,0.00001,1024,0.855248,0.635964,0.634757,0.639584,0.637161,0.660039,3196,3163,1839,1801
4,5,0.00005,0.00001,1024,0.850270,0.621962,0.650507,0.526316,0.581858,0.648140,2630,3589,1413,2367
5,6,0.00010,0.00001,1024,0.849959,0.638964,0.632676,0.661797,0.646909,0.653092,3307,3082,1920,1690
6,7,0.00005,0.00001,1024,0.843822,0.633763,0.630244,0.646388,0.638214,0.654954,3230,3107,1895,1767
7,8,0.00010,0.00001,1024,0.850356,0.634063,0.647033,0.589153,0.616738,0.653734,2944,3396,1606,2053
8,9,0.00010,0.00001,1024,0.850960,0.632763,0.657851,0.552532,0.600609,0.655972,2761,3566,1436,2236
9,10,0.00050,0.00001,1024,0.852168,0.637564,0.640705,0.625575,0.633050,0.652618,3126,3249,1753,1871


In [14]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureB_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df

,metric,mean,std,standard_error
0,accuracy,0.634473,0.008747,0.002766
1,precision,0.639831,0.009048,0.002861
2,recall,0.616830,0.057791,0.018275
3,f1,0.626549,0.027433,0.008675
4,auc,0.654229,0.008270,0.002615


In [15]:
# Best learning rate frequency for FT-Transformer

best_lr_frequency = (
    results_df["best_lr"]
    .value_counts()
    .reset_index()
)

best_lr_frequency.columns = ["learning_rate", "frequency"]

best_lr_frequency_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureB_best_lr_frequency.csv"
)

best_lr_frequency.to_csv(
    best_lr_frequency_path,
    index=False,
    encoding="utf-8-sig"
)


best_lr_frequency

,learning_rate,frequency
0,0.00010,5
1,0.00005,3
2,0.00050,2
